# Inductance simulation — resonator → SQUID-array flux coupling

Inject a DC current at the **open end** of the readout resonator and integrate the
magnetic flux through the **interior of every SQUID loop** in `array_1`.

`flux_per_amp = Σᵢ Φᵢ / I` is a purely **geometric** mutual inductance: the DC
current is *uniform* along the resonator, not the λ/4 mode profile. Use it to
**rank array geometries** for inductive coupling — it is not the true mode–array
coupling (that also weights the resonator current profile).

The JJ gaps are left open (no lumped port), so each loop is an open ring that
carries no screening supercurrent. The array metal is still PEC, and it matters:
its arms exclude B and funnel ~22% *more* flux through the openings than bare
vacuum apertures would (measured — see the solver note). So the array stays in
the model even though it is what makes the solve slow.

The design comes from `build_design.build_design()`; edit parameters there or pass
overrides to that call, then re-run from the top.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from shapely.ops import unary_union
from shapely.affinity import scale

from qiskit_metal import MetalGUI
from SQDMetal.Utilities.QUtilities import QUtilities

from build_design import build_design, ARRAY_COMPONENT_NAME, N_SQUIDS

# Chip is sized to the geometry + an ABSOLUTE margin (not the 2.5x factor): the
# hairpin field is a localised 1/r^2 dipole, so a few hundred um of ground plane
# past the metal is plenty. Substrate thickness is magnetostatically inert
# (mu_r = 1), so it is thinned purely to shrink the mesh. NB: the far-field
# volume barely drives RAM here -- the fine mesh at the loops does (see below).
CHIP_MARGIN_UM = 250
CHIP_Z_UM      = 200

# pass geometry overrides too, e.g. build_design(squid_gap_um=20, arm_semi_length_um=12)
design = build_design(chip_margin_um=CHIP_MARGIN_UM, chip_size_z_um=CHIP_Z_UM)
design.rebuild()

gui = MetalGUI(design)
gui.rebuild()
gui.autoscale()

sz = design.chips.main.size
print("components:", list(design.components))
print(f"n_squids = {N_SQUIDS}")
print(f"chip = {sz.size_x} x {sz.size_y} x {sz.size_z}")

## Integration area = union of all SQUID-loop openings

`SquidArray` already emits every loop opening as a `subtract=True` poly named
`loop_interiors_*` (metal-bounded rectangle, `(seg_a + seg_b + JJ_gap) × squid_gap`).
We just collect them and convert qiskit-metal's mm to the metres that
`add_integration_area` expects. No `PVD_Shadows` needed — there is no evaporation
layer in this design.

In [ ]:
def squid_loop_interiors(design, array_name, inset_um=0.0):
    """Union of every SQUID-loop opening in a SquidArray, in METRES.

    inset_um > 0 shrinks every loop inward — use only if the mesher chokes on the
    integration-surface edges coinciding with the array metal edges.
    """
    poly = design.qgeometry.tables['poly']
    aid = design.components[array_name].id
    rows = poly[(poly['component'] == aid)
                & poly['name'].str.startswith('loop_interiors')]
    if len(rows) == 0:
        raise RuntimeError(f"no 'loop_interiors*' polys on component {array_name!r}")
    geom_mm = unary_union(list(rows['geometry']))
    u = QUtilities.get_units(design)                      # metres per qiskit unit (1e-3)
    geom_m = scale(geom_mm, xfact=u, yfact=u, origin=(0, 0))
    if inset_um:
        geom_m = geom_m.buffer(-inset_um * 1e-6)
    return geom_m


int_area = squid_loop_interiors(design, ARRAY_COMPONENT_NAME)
n_loops = len(int_area.geoms)
print(f"{n_loops} loops,  total area = {int_area.area:.3e} m^2"
      f"  ({int_area.area / n_loops:.3e} m^2 per loop)")

In [ ]:
# sanity sketch: array metal (grey) + flux integration surfaces (red)
fig, ax = plt.subplots(figsize=(11, 3))
tbl = design.qgeometry.tables['poly']
aid = design.components[ARRAY_COMPONENT_NAME].id
u = QUtilities.get_units(design)
for g in tbl[(tbl['component'] == aid) & (~tbl['subtract'])]['geometry']:
    for gg in getattr(g, 'geoms', [g]):
        ax.fill(*gg.exterior.xy, color='0.75', lw=0)
for g in int_area.geoms:
    gg = scale(g, 1 / u, 1 / u, origin=(0, 0))            # metres -> mm for plotting
    ax.fill(*gg.exterior.xy, color='tab:red', alpha=0.6, lw=0)
ax.set_aspect('equal')
ax.set_xlabel('x (mm)'); ax.set_ylabel('y (mm)')
ax.set_title('SQUID-array metal (grey) + flux integration surfaces (red)')
plt.show()

## PALACE magnetostatic simulation

### Work around an SQDMetal limitation

`add_integration_area` only handles a **single `Polygon`**. Our integration area is
a `MultiPolygon` (23 disjoint loops); where it clips the one connected dielectric
gap around the array, `GMSH_Geometry_Builder` ends up with a `MultiPolygon` gap
piece and crashes on `.exterior` (and passing the 23 loops as separate areas hits
a name-collision bug in the area subdivision — they all collapse to one loop).

The cell below monkey-patches `_create_gmsh_geometry_from_shapely_polygons` to
flatten any `MultiPolygon` in its input into its constituent `Polygon`s (exactly
what the surrounding code already does for the "left-over" pieces). One integration
area with 23 sub-surfaces then becomes **one** Palace `SurfaceFlux` whose attribute
list covers all 23 loops, i.e. Palace returns the summed flux `Σᵢ Φᵢ`. Verified:
the resulting physical group holds 23 surfaces totalling the expected loop area.

In [ ]:
from SQDMetal.PALACE.Utilities.GMSH_Geometry_Builder import GMSH_Geometry_Builder
from SQDMetal.Utilities.ShapelyEx import ShapelyEx

_orig_mk = GMSH_Geometry_Builder._create_gmsh_geometry_from_shapely_polygons

def _mk_flatten_multipolys(self, polygons):
    flat = []
    for p in polygons:
        flat.extend(list(p.geoms) if p.geom_type == 'MultiPolygon' else [p])
    return _orig_mk(self, flat)

GMSH_Geometry_Builder._create_gmsh_geometry_from_shapely_polygons = _mk_flatten_multipolys
print("patched GMSH_Geometry_Builder._create_gmsh_geometry_from_shapely_polygons")

### Keeping the mesh (and RAM) small

Element count here is set almost entirely by **where the fine mesh lives**, not by
the chip size — shrinking the chip footprint barely moved the tet count in tests,
while the `min_size` of the loop-region fine mesh moved it ~10x.

So instead of fine-meshing the whole `array_1` metal + both terminations + the
2.4 mm readout trace, this fine-meshes a **thin ribbon** covering just the loop
band (the array x-extent, y a hair past the loop metal). That ribbon contains
every loop opening (the integration surfaces) and the array arms; the resonator,
carrying a uniform DC current far from the loops, gets the default coarse mesh.

Knobs, roughly in order of leverage on RAM:

| knob | effect |
|---|---|
| `LOOP_MESH_UM` (ribbon `min_size`) | 2.5 → ~10 elems across a 25 µm loop; 2.0 ≈ 2x the tets, 3.0 ≈ 0.7x |
| `solver_order` (kept at 1) | order 2 is ~5x more DoF **and** its CG+AMS stalls here — see the solver-stall note below |
| `RIBBON_TAPER_UM` | how fast elements coarsen leaving the ribbon |
| chip / air box | minor here; kept modest for a clean field boundary |

With the ribbon mesh the solve is ~0.7 GB over 8 ranks — RAM is no longer the
constraint. The remaining issue was **linear-solver convergence**: with the array
present, Palace's default CG + AMS stalls (residual plateaus ~2e-2) because the
chain is ~24 floating thin PEC islands cut apart by the open JJ gaps — a
near-null-space CG cannot climb out of. The fix is **GMRES** instead of CG
(patched into the config below); it grinds — ~570 iterations, reduction factor
~0.98 — but converges in ~1 min for this mesh.

In [ ]:
from SQDMetal.PALACE.Inductance_Simulation import PALACE_Inductance_Simulation

LOOP_MESH_UM    = 2.5     # element size in the loop band (integration surfaces + array arms)
RIBBON_TAPER_UM = 6.0     # distance the mesh stays at LOOP_MESH_UM before coarsening
RIBBON_PAD_UM   = 8.0     # how far the ribbon extends past the loop-opening bbox

user_defined_options = {
    "mesh_refinement":     0,          # no global uniform mesh doubling
    "dielectric_material": "silicon",  # inert for magnetostatics (mu_r = 1), kept for the API
    "solver_order":        1,          # order 1; the field feeding the flux integral is smooth.
                                       #   order 2 is 5x the DoF for little gain here and needs
                                       #   GMRES just the same.
    "solver_tol":          1.0e-6,     # residual tol; the integrated flux is converged long
                                       #   before this
    "solver_maxits":       800,        # GMRES grinds on this geometry (~570 iters) -- give it room
    "fillet_resolution":   8,          # vertices per 90 deg when polygonising filleted paths
    "palace_dir":          "/home/ruiz/repo/spack/opt/spack/linux-zen3/palace-develop-blpbtxe43ne2or3yo7gou6ycmdwdxuzl/bin/palace",
    "num_cpus":            8,
}

mag_sim = PALACE_Inductance_Simulation(
    name='ind_squid_array',
    metal_design=design,
    sim_parent_directory="",           # output dir ./ind_squid_array/ next to this notebook
    mode='simPC',
    meshing='GMSH',
    user_options=user_defined_options,
    create_files=True,
)

mag_sim.add_metallic(1)                 # everything is on layer 1 (no evaporation layer)
mag_sim.add_ground_plane()             # applies array_1's ground_pocket + loop_interiors cutouts

# DC current in at the readout OPEN end (readout_1 'start' == launch_point1); it
# returns via launch_point2 (ShortToGround) through the ground plane. The U-clip
# ties the ground planes around the injection pin to close the DC loop.
mag_sim.create_current_source_with_Uclip_on_Route('readout_1', 'start')

# one surface spanning every SQUID loop -> Palace returns the SUMMED flux  sum_i Phi_i
mag_sim.add_integration_area(int_area)

# trim the far-field box (small win, and keeps the PEC wall a clean distance off)
mag_sim.set_xBoundary_as_proportion(0.05)
mag_sim.set_yBoundary_as_proportion(0.05)
mag_sim.set_zBoundary_as_proportion(z_prop_top=1.5, z_prop_bottom=0.0)

# fine mesh ONLY the loop band: a ribbon over the loop-opening bbox + a little pad
x1, y1, x2, y2 = int_area.bounds          # already in metres
pad = RIBBON_PAD_UM * 1e-6
mag_sim.fine_mesh_in_rectangle(x1 - pad, y1 - pad, x2 + pad, y2 + pad,
                               min_size=LOOP_MESH_UM * 1e-6, max_size=60e-6,
                               taper_dist_min=RIBBON_TAPER_UM * 1e-6)
# a lighter pass on the array metal edges so the PEC outline is still clean
mag_sim.fine_mesh_components(['array_1'], min_size=1.5 * LOOP_MESH_UM * 1e-6,
                             max_size=60e-6, taper_dist_min=6e-6, metals_only=True)

mag_sim.prepare_simulation()

In [ ]:
# REQUIRED: switch the Krylov solver CG -> GMRES. SQDMetal hardcodes CG in the
# generated config; CG stalls here (floating PEC islands). Must run after
# prepare_simulation() and before run().
import json

_cfg_path = mag_sim._sim_config
_cfg = json.load(open(_cfg_path))
_cfg["Solver"]["Linear"]["KSPType"] = "GMRES"
json.dump(_cfg, open(_cfg_path, "w"), indent=2)
print("Solver.Linear:", _cfg["Solver"]["Linear"])

In [ ]:
# element count -- a decent RAM proxy before committing to the solve
import gmsh
n_tet = len(gmsh.model.mesh.getElementsByType(4)[0])
n_nod = len(gmsh.model.mesh.getNodes()[0])
print(f"tetrahedra: {n_tet:,}   nodes: {n_nod:,}")
print(f"~solve DoF: {(7 if user_defined_options['solver_order'] == 1 else 38) * n_nod / 1e6:.1f}M "
      f"(order {user_defined_options['solver_order']}, H(curl))")

In [ ]:
mag_sim.open_mesh_gmsh()   # inspect: every loop opening + the short termination must be resolved

### Why ~570 iterations (and why not fewer)

The ~23 open SQUID loops are topological handles in the field domain — ~23
harmonic near-null modes AMS can't precondition, which is what forces GMRES over
CG and gives the poor ~0.97 per-iteration reduction. It's not a mesh problem;
refining makes it worse. It still converges to the right answer in ~1 min.

**Dropping the array metal was tested on the cluster and rejected**: with the
metal *out* (`build_design(include_array_metal=False)`, loop apertures = vacuum in
a `RectCutout` window) the solve is 2-3x faster (~340 CG-ish iterations) but the
flux comes out **~22% low** (9.8 vs 12.5 Φ₀/mA) — the PEC arms exclude B and
funnel extra flux through the openings, and that concentration depends on loop
geometry, so it is not common-mode across a sweep. Keep the array in.

If GMRES itself flat-lines: raise `solver_maxits` (2000), loosen `solver_tol` to
1e-5 (the integrated flux is converged to 4+ digits long before), or coarsen
`LOOP_MESH_UM` to 3.

### Run

One linear magnetostatic solve, ~1 min / ~0.7 GB over 8 ranks with the settings
above. To sweep geometries, loop `build_design(...)` -> rebuild `int_area` -> new
`PALACE_Inductance_Simulation(name=...)` per point.

Old note (no longer applies): the 2 µm whole-array mesh that made this memory-heavy
has been replaced by the loop-band ribbon.

In [ ]:
flux_per_amp = mag_sim.run()          # (sum_i Phi_i) / I   [Wb/A], one number
flux_per_amp

In [ ]:
PHI0 = 2.067833848e-15   # Wb, flux quantum
M = float(np.atleast_2d(flux_per_amp).sum())    # total flux linkage per amp == mutual inductance [H]

print(f"total flux linkage / current : {M:.4e} Wb/A   (= M, henries)")
print(f"                            : {M / PHI0 * 1e-3:.4e} Phi_0 / mA")
print(f"per loop (average)          : {M / N_SQUIDS / PHI0 * 1e-3:.4e} Phi_0 / mA")

### Per-loop flux profile

Palace returns only the **summed** flux here (one `SurfaceFlux` over all loops).
Seeing the flux loop-by-loop (a uniformity check) would need a real fix to
SQDMetal's area subdivision: registering the loops as separate integration areas
currently collapses them to one, because the `+'i'` piece names it generates are
not unique per area. Until then a per-loop profile means one solve per loop with a
single-`Polygon` `int_area` each -- only worth doing as an occasional sanity check
on one geometry, never in the sweep.